In [1]:
from openai import OpenAI
import pandas as pd
import minsearch


In [2]:
df = pd.read_csv('../data/cleaned_data.csv')

In [3]:

documents = df.to_dict(orient='records')

In [4]:
index = minsearch.Index(['task', 'category', 'difficulty', 'duration_estimate'
       'framework_name', 'reasoning', 'instructions', 'tags'],
        keyword_fields=["id"])


In [5]:
index.fit(documents)

In [6]:
client = OpenAI()

In [7]:
def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [8]:
prompt_template = """
You're a productivity advisor. Answer the QUESTION based on the CONTEXT from our productivity tasks database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
task: {task}
category: {category}
difficulty: {difficulty}
duration_estimate: {duration_estimate}
instructions: {instructions}
reasoning: {reasoning}
tags: {tags}
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt


In [9]:
def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content


In [10]:
def rag(query, model='gpt-4o-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, model=model)
    return answer

In [11]:
question = "How can I organize my workspace efficiently using a structured method?"
answer = rag(question)
print(answer)

To organize your workspace efficiently using a structured method, you can follow these steps based on the context above:

1. **Clear Your Desk**: Start by clearing everything off your desk to create a clean slate. This can also help you assess what is essential.

2. **Sort Items**: Group items into categories, such as documents, stationery, and personal items. This is similar to the GTD (Getting Things Done) method, which helps in categorizing clutter.

3. **Prioritize Essentials**: Determine what items you use regularly and keep only those on your desk. Place the essentials prominently, while less frequently used items can be stored away.

4. **Organize**: Use containers or folders to group similar items. For example, you might sort supplies, label containers, and arrange them neatly, ensuring everything has its designated place.

5. **Maintain**: Regularly review your workspace and repeat the sorting and organizing process to avoid clutter accumulation, just like scheduling time to c

In [ ]:
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')

In [9]:
from pydantic import BaseModel, Field
from typing import List

class Task(BaseModel):
    task: str = Field(description="Name or title of the task")
    category: str = Field(description="Category such as work, home, fitness, study, etc.")
    difficulty: str = Field(description="Difficulty level: easy, medium, hard")
    duration_estimate: int = Field(description="Estimated duration in minutes")
    instructions: str = Field(description="Step-by-step instructions for completing the task")
    reasoning: str = Field(description="Explanation of why this task maps to a specific framework")
    tags: str = Field(description="Comma-separated tags for search and filtering")


In [10]:
class TaskDataset(BaseModel):
    tasks: List[Task]

In [12]:
openai_client = OpenAI()
prompt = """
Generate a dataset of 50 diverse productivity tasks.
Cover different categories such as home, work, fitness, study, personal, and creative.
Vary difficulty levels and duration estimates.
Each task must include: task, category, difficulty, duration_estimate, instructions, reasoning, tags.
Return the dataset in the exact schema provided.
""".strip()
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=[{"role": "user", "content": prompt}],
    text_format=TaskDataset,
)

dataset = response.output_parsed
df = pd.DataFrame([task.model_dump() for task in dataset.tasks])
df.to_csv("../data/tasks.csv", index=False)
print(f"Generated {len(df)} tasks")


Generated 53 tasks


In [13]:
q = "Find a task that teaches me how to improve my focus using a structured method."

In [14]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": q}]
)
response.choices[0].message.content 

"Improving focus is a valuable skill that can be cultivated through a structured method. Here’s a task that utilizes the Pomodoro Technique, a popular time management method designed to enhance concentration and productivity.\n\n### Task: Implement the Pomodoro Technique\n\n**Objective:** To improve focus and productivity through time management.\n\n#### Step 1: Prepare Your Workspace\n- **Choose a quiet environment** free of distractions. \n- **Remove any clutter** from your desk or workspace.\n- **Gather necessary materials**, such as paper, pens, or digital tools you will need for your task.\n\n#### Step 2: Select a Task\n- Choose a specific task or project you want to focus on. Ensure it's something that can be accomplished in segments.\n\n#### Step 3: Set Up a Timer\n- Use a timer, phone app, or an online Pomodoro timer.\n- **Set the timer for 25 minutes.** This is one Pomodoro session.\n\n#### Step 4: Work on the Task\n- Start working on your chosen task as soon as the timer star

In [ ]:


# Load dataset
df = pd.read_csv("../data/data.csv")   # notebook is in sibling folder

# Convert rows to dicts
documents = df.to_dict(orient="records")

# Build MinSearch index
index = Index(
    text_fields=[
        "task",
        "instructions",
        "reasoning",
        "tags",
        "category",
        "difficulty"
    ],
    keyword_fields=[
        "duration_estimate"
    ]
)

# Fit index
index.fit(documents)
